# RAG Pipline  - Data Ingestion To Vectoer DB

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader , PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from pathlib import Path

import os


c:\Users\Lenovo\OneDrive\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Jake_s_Resume.pdf
  ✓ Loaded 1 pages

Processing: Maged_Yasser_CV.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-11T13:09:01+00:00', 'author': '', 'keywords': '', 'moddate': '2025-09-11T13:09:01+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Jake_s_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Jake_s_Resume.pdf', 'file_type': 'pdf'}, page_content='Maged Yasser\n+20 150212904 | magedyasser000@gmail.com | linkedin.com/in/maged-yasser | github.com/magedyasse\nEducation\nMansoura University Mansoura, Egypt\nB.Sc. in Science and Technology, Dept. of Artificial Intelligence 2022 – Present\n• 3rd place in first year, 2nd place in second year.\n• Won 1st place in AI semester competition by building a multi-agent AI system.\nExperience\nMachine Learning Trainee 2024\nNTI (National Telecommunication Institute) Egypt\n• Learned

In [4]:
def split_documents(documents, chunk_size=700, chunk_overlap=100):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", "","\n• " ,  "\n- "]
    )
    
    
    split_doc = text_splitter.split_documents(documents)
    print(f"Split  {len(documents)} documents into {len(split_doc)} chunks"
          f" and each chunk size is {chunk_size} with overlap of {chunk_overlap}")
    
    
    if split_doc:
       print("Example split document chunk: ")
       print(split_doc[0].page_content[:200])  # Print first 100 characters of the first chunk
       print("Metadata: ", split_doc[0].metadata)
       
    return split_doc    
    
    
    
    # all_splitted_docs = []
    # for doc in documents:
    #     splits = text_splitter.split_documents([doc])
    #     all_splitted_docs.extend(splits)
    
    # print(f"Total documents after splitting: {len(all_splitted_docs)}")
    # return all_splitted_docs

In [5]:
chunked_pdf_documents = split_documents(all_pdf_documents)
chunked_pdf_documents

Split  2 documents into 7 chunks and each chunk size is 700 with overlap of 100
Example split document chunk: 
Maged Yasser
+20 150212904 | magedyasser000@gmail.com | linkedin.com/in/maged-yasser | github.com/magedyasse
Education
Mansoura University Mansoura, Egypt
B.Sc. in Science and Technology, Dept. of Art
Metadata:  {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-11T13:09:01+00:00', 'author': '', 'keywords': '', 'moddate': '2025-09-11T13:09:01+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Jake_s_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Jake_s_Resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-11T13:09:01+00:00', 'author': '', 'keywords': '', 'moddate': '2025-09-11T13:09:01+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Jake_s_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Jake_s_Resume.pdf', 'file_type': 'pdf'}, page_content='Maged Yasser\n+20 150212904 | magedyasser000@gmail.com | linkedin.com/in/maged-yasser | github.com/magedyasse\nEducation\nMansoura University Mansoura, Egypt\nB.Sc. in Science and Technology, Dept. of Artificial Intelligence 2022 – Present\n• 3rd place in first year, 2nd place in second year.\n• Won 1st place in AI semester competition by building a multi-agent AI system.\nExperience\nMachine Learning Trainee 2024\nNTI (National Telecommunication Institute) Egypt\n• Learned

# Embeddings Vectoer Store DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List ,Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingVectorStore:
    
    """This class handles embedding generation and storage in ChromaDB
    steps involved:
    1. Initialize embedding model and ChromaDB client
    2. Embed texts using the embedding model
    3. Add documents along with their embeddings to ChromaDB
    4. Query the vector store to retrieve similar documents based on a query text
    """
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        
        """
        Initialize the embedding model and ChromaDB client. 
        
        args :
            model_name (str): Name of the sentence transformer model to use for embeddings.
        """
        
        self.model_name = model_name
        self.model= None
        self._load_model()
        
    
    def _load_model(self):
        """
        Load the sentence transformer model .
        
        args:
            model_name (str): Name of the sentence transformer model to load.
        """
        
        try :
            print(f"Loading embedding model: {self.model_name} ...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. "
                  f"with embedding dimension {self.model.get_sentence_embedding_dimension()}")
            
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")  
            raise   
        
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        
        args:
            texts (List[str]): List of texts to embed.
        """
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise    
        
    def get_dimension(self) -> int:
        """
        Get the embedding dimension of the model.
        
        returns:
            int: Dimension of the embeddings.
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        else :
            return self.model.get_sentence_embedding_dimension()    

In [8]:
embedding_manager = EmbeddingVectorStore()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2 ...
Model all-MiniLM-L6-v2 loaded successfully. with embedding dimension 384


## Vectoer Store

In [70]:
class Vectoer_Store:
    """
    Here  Is the Vector Store Class to handle all the operations related to vector store DB
    1. Initialize ChromaDB Client and Collection
    2. Add documents with embeddings to the collection
    3. Query the collection for similar documents
    """
    
    def __init__(self, collection_name: str = "pdf_documents_collection", persist_directory: str = "../data/vector_store_db"):
        """
        Initialize ChromaDB client and collection.
        
        args:
            collection_name (str): Name of the ChromaDB collection.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
        
        
    def _initialize_chromadb(self):
        """
        Initialize ChromaDB client and collection.
        """
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
                
            # Use cosine distance for better similarity matching
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings",
                         "hnsw:space": "cosine"})  # Use cosine distance
            
            print(f"ChromaDB collection '{self.collection_name}' initialized successfully")
            print(f"Distance metric: cosine")
            print(f"Existing documents in the collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise    
        
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents with embeddings to the ChromaDB collection.
        
        args:
            documents (List[Any]): List of document objects to add.
            embeddings (np.ndarray): Array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        
        print(f"Adding {len(documents)} documents to the collection...")
        
        # prepare data for insertion ChromaDB
        ids = [] 
        metadatas = []
        doc_texts = []
        embedings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            
            # prepare metadata
            metadata = dict(doc.metadata)  # Copy existing metadata
            metadata['doc_index'] = i  # Add document index to metadata
            metadatas.append(metadata)
            
            # prepare embedding
            doc_texts.append(doc.page_content)
            
            
            # prepare embedding
            embedings_list.append(embedding.tolist())
            
        # add to chromaDB collection
        try:
            
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=doc_texts,
                embeddings=embedings_list
            )
            print(f"Successfully added {len(documents)} documents to the collection.")
            print(f"Total documents in the collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to ChromaDB: {e}")
            raise

In [71]:
vectoer_Store = Vectoer_Store()
vectoer_Store

ChromaDB collection 'pdf_documents_collection' initialized successfully
Distance metric: cosine
Existing documents in the collection: 7


In [81]:
# Delete the old collection and recreate with cosine distance
try:
    vectoer_Store.client.delete_collection("pdf_documents_collection")
    print("✓ Old collection deleted successfully")
except Exception as e:
    print(f"Collection deletion: {e}")

# Recreate the vector store with cosine distance
vectoer_Store = Vectoer_Store()
print("\n✓ New vector store created with cosine distance")

✓ Old collection deleted successfully
ChromaDB collection 'pdf_documents_collection' initialized successfully
Distance metric: cosine
Existing documents in the collection: 0

✓ New vector store created with cosine distance


In [82]:
chunked_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-11T13:09:01+00:00', 'author': '', 'keywords': '', 'moddate': '2025-09-11T13:09:01+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Jake_s_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Jake_s_Resume.pdf', 'file_type': 'pdf'}, page_content='Maged Yasser\n+20 150212904 | magedyasser000@gmail.com | linkedin.com/in/maged-yasser | github.com/magedyasse\nEducation\nMansoura University Mansoura, Egypt\nB.Sc. in Science and Technology, Dept. of Artificial Intelligence 2022 – Present\n• 3rd place in first year, 2nd place in second year.\n• Won 1st place in AI semester competition by building a multi-agent AI system.\nExperience\nMachine Learning Trainee 2024\nNTI (National Telecommunication Institute) Egypt\n• Learned

In [83]:
#  Convert document chunks to texts for embedding
texts_to_embed = [doc.page_content for doc in chunked_pdf_documents]
texts_to_embed

['Maged Yasser\n+20 150212904 | magedyasser000@gmail.com | linkedin.com/in/maged-yasser | github.com/magedyasse\nEducation\nMansoura University Mansoura, Egypt\nB.Sc. in Science and Technology, Dept. of Artificial Intelligence 2022 – Present\n• 3rd place in first year, 2nd place in second year.\n• Won 1st place in AI semester competition by building a multi-agent AI system.\nExperience\nMachine Learning Trainee 2024\nNTI (National Telecommunication Institute) Egypt\n• Learned ML algorithms: Linear Regression, Logistic Regression.\n• Hands-on with Docker, FastAPI, Gradio, Streamlit, HuggingFace, Pydantic, and MLOps concepts.\nMachine Learning Trainee 2023\nMansoura University Mansoura, Egypt',
 'Machine Learning Trainee 2023\nMansoura University Mansoura, Egypt\n• Studied SVM, Decision Trees, PCA, and Introduction to Deep Learning.\nProjects\nHeart Failure Prediction | Python, FastAPI, Gradio, Streamlit, Sklearn, Imblearn, Docker 2024\n• Built ML system for predicting heart failure risk

In [84]:
# Generate embeddings for the chunked documents
embeddings = embedding_manager.generate_embeddings(texts_to_embed)

Generating embeddings for 7 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.32it/s]

Generated embeddings with shape: (7, 384)


In [85]:
# store in vector DB
vectoer_Store.add_documents(chunked_pdf_documents , embeddings)

Adding 7 documents to the collection...
Successfully added 7 documents to the collection.
Total documents in the collection: 7


# RAG Pipeline From Vectoer Store

In [86]:
class RAGRetrieval:
    
    """RAG Pipeline to handle retrieval and generation using vector store and LLM
    1. Initialize with vector store and LLM model
    2. Retrieve relevant documents based on query
    3. Generate response using LLM based on retrieved documents
    """
    
    def __init__(self, vector_store: Vectoer_Store, embedding_manager: EmbeddingVectorStore):
        """
        Initialize RAGRetrieval with vector store and LLM model.
        
        args:
            vector_store (Vectoer_Store): Instance of the vector store.
            embedding_manager (EmbeddingVectorStore): Instance of the embedding manager.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
     
    
    def retrieve_documents(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:   
        """
        Retrieve relevant documents from the vector store based on the query.
        
        args:
            query (str): The input query text.
            top_k (int): Number of top documents to retrieve.
            score_threshold (float): Minimum similarity score threshold for retrieval.
        returns:
            List[Dict[str, Any]]: List of retrieved documents with metadata.
        """
        print(f"Retrieving top {top_k} documents for query: '{query}'")
        print(f"Using score threshold: {score_threshold}")
    
        # Generate query embedding
        try:
            query_embeddings = self.embedding_manager.generate_embeddings([query])
            query_embedding = query_embeddings[0].tolist()
            print(f"Query embedding generated successfully, shape: {query_embeddings.shape}")
        except Exception as e:
            print(f"Error generating query embedding: {e}")
            import traceback
            traceback.print_exc()
            return []
        
        
        # search in vector store
        try:
            
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k
            )
            
            print(f"Query results structure: {type(results)}")
            print(f"Has documents: {bool(results.get('documents'))}")
            
            # Filter results based on score threshold
            retrieved_docs = []
            
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                print(f"Found {len(documents)} documents before filtering")
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    
                    # Convert distance to similarity score
                    # For cosine distance in ChromaDB: similarity = 1 - distance
                    # Distance is already in range [0, 2], where 0 = identical, 2 = opposite
                    similarity_score = 1 - distance
                    
                    print(f"Doc {i+1}: distance={distance:.4f}, similarity={similarity_score:.4f}")
                    
                    if similarity_score >= score_threshold:
                        # Add content_length to metadata if not present
                        if 'content_length' not in metadata:
                            metadata['content_length'] = len(document)
                            
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold.")
            else:
                print("No documents retrieved from the vector store.")  
            
            return retrieved_docs
                
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            import traceback
            traceback.print_exc()
            return []

In [87]:
rag_retrieval = RAGRetrieval(vector_store=vectoer_Store , embedding_manager= embedding_manager)

In [88]:
rag_retrieval

In [89]:
rag_retrieval.retrieve_documents(query="github" )

Retrieving top 5 documents for query: 'github'
Using score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 103.37it/s]

Generated embeddings with shape: (1, 384)
Query embedding generated successfully, shape: (1, 384)
Query results structure: <class 'dict'>
Has documents: True
Found 5 documents before filtering
Doc 1: distance=0.6551, similarity=0.3449
Doc 2: distance=0.6858, similarity=0.3142
Doc 3: distance=0.7495, similarity=0.2505
Doc 4: distance=0.8422, similarity=0.1578
Doc 5: distance=0.8489, similarity=0.1511
Retrieved 5 documents after applying score threshold.


[{'id': 'doc_7b18d286_6',
  'content': 'Technical Skills\nLanguages: Python, SQL, HTML, CSS\nFrameworks: FastAPI, Streamlit, Gradio\nLibraries: Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, PyTorch, TensorFlow\nDevOps & Tools: Docker, Git, GitHub, VS Code\nOther: Prompt Engineering, Data Storytelling, Web Scraping, Data-Driven Decision Making',
  'metadata': {'source_file': 'Maged_Yasser_CV.pdf',
   'creationdate': '2025-09-13T23:11:50+00:00',
   'total_pages': 1,
   'page_label': '1',
   'keywords': '',
   'source': '..\\data\\pdf\\Maged_Yasser_CV.pdf',
   'creator': 'LaTeX with hyperref',
   'producer': 'pdfTeX-1.40.26',
   'moddate': '2025-09-13T23:11:50+00:00',
   'file_type': 'pdf',
   'author': '',
   'doc_index': 6,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0',
   'trapped': '/False',
   'page': 0,
   'title': '',
   'subject': '',
   'content_length': 303},
  'similarity_score': 0.34494495391845703,
  'distan

#  Simple RAG Pipeline With Groq LLM

In [90]:
from langchain_groq import ChatGroq
import os  
from dotenv import load_dotenv
load_dotenv()

True

In [100]:
grog_api_key = "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"

# Use a currently supported model
# Options: "llama-3.3-70b-versatile", "llama-3.1-8b-instant", "mixtral-8x7b-32768"
llm = ChatGroq(
    groq_api_key=grog_api_key, 
    model_name="llama-3.3-70b-versatile",  # Updated to supported model
    temperature=0.1, 
    max_tokens=1024
)

In [101]:
# simple test
def rag_simple(query, retriever, llm, top_k=3):
    """
    Simple RAG function to retrieve documents and generate an answer using LLM.
    
    Args:
        query (str): User's question
        retriever: RAGRetrieval instance
        llm: Language model instance
        top_k (int): Number of documents to retrieve
    
    Returns:
        str: Generated answer
    """
    
    # Retrieve relevant documents
    results = retriever.retrieve_documents(query=query, top_k=top_k, score_threshold=0.1)
    
    # Build context from retrieved documents
    if results:
        context = "\n\n".join([doc['content'] for doc in results])
    else:
        return "No relevant documents found. I cannot answer this question based on the available documents."
    
    # Create prompt with context and query
    prompt = f"""You are a helpful assistant. Use ONLY the following context to answer the question. If the answer cannot be found in the context, say "I cannot find this information in the provided documents."

Context:
{context}

Question: {query}

Answer:"""
    
    # Generate response using LLM
    try:
        response = llm.invoke(prompt)
        return response.content
    except Exception as e:
        return f"Error generating response: {e}"

In [102]:
answer = rag_simple("what the Number in doc" , rag_retrieval , llm  , top_k= 3)
print(answer)

Retrieving top 3 documents for query: 'what the Number in doc'
Using score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 75.18it/s]

Generated embeddings with shape: (1, 384)
Query embedding generated successfully, shape: (1, 384)
Query results structure: <class 'dict'>
Has documents: True
Found 3 documents before filtering
Doc 1: distance=0.8473, similarity=0.1527
Doc 2: distance=0.8621, similarity=0.1379
Doc 3: distance=0.8833, similarity=0.1167
Retrieved 3 documents after applying score threshold.


I cannot find this information in the provided documents.


In [103]:
# Test with more specific queries
test_queries = [
    "What is Maged's phone number?",
    "What is the contact number?",
    "Tell me about the phone number or email",
    "What numbers are mentioned in the document?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    answer = rag_simple(query, rag_retrieval, llm, top_k=3)
    print(f"Answer: {answer}\n")


Query: What is Maged's phone number?
Retrieving top 3 documents for query: 'What is Maged's phone number?'
Using score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.42it/s]

Generated embeddings with shape: (1, 384)
Query embedding generated successfully, shape: (1, 384)
Query results structure: <class 'dict'>
Has documents: True
Found 3 documents before filtering
Doc 1: distance=0.7390, similarity=0.2610
Doc 2: distance=0.7605, similarity=0.2395
Doc 3: distance=0.8953, similarity=0.1047
Retrieved 3 documents after applying score threshold.


Answer: +20 150212904


Query: What is the contact number?
Retrieving top 3 documents for query: 'What is the contact number?'
Using score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.42it/s]



...

Generated embeddings with shape: (1, 384)
Query embedding generated successfully, shape: (1, 384)
Query results structure: <class 'dict'>
Has documents: True
Found 3 documents before filtering
Doc 1: distance=0.8817, similarity=0.1183
Doc 2: distance=0.8849, similarity=0.1151
Doc 3: distance=0.9162, similarity=0.0838
Retrieved 2 documents after applying score threshold.


Answer: I cannot find this information in the provided documents.


Query: What numbers are mentioned in the document?
Retrieving top 3 documents for query: 'What numbers are mentioned in the document?'
Using score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.31it/s]

Generated embeddings with shape: (1, 384)
Query embedding generated successfully, shape: (1, 384)
Query results structure: <class 'dict'>
Has documents: True
Found 3 documents before filtering
Doc 1: distance=0.8905, similarity=0.1095
Doc 2: distance=0.9015, similarity=0.0985
Doc 3: distance=0.9163, similarity=0.0837
Retrieved 1 documents after applying score threshold.


Answer: I cannot find specific numbers mentioned in the provided documents, except for the fact that there is a mention of "Year" in the context of certifications, but no specific year is provided.



In [55]:
# # Diagnostic: Check vector store status
# print(f"Total documents in collection: {vectoer_Store.collection.count()}")
# print(f"\nFirst 3 documents in collection:")
# sample = vectoer_Store.collection.peek(limit=3)
# for i, doc in enumerate(sample['documents']):
#     print(f"\n--- Document {i+1} ---")
#     print(doc[:200])  # First 200 characters

In [56]:
# # Test with a simpler direct query to ChromaDB
# query_text = "Maged Yasser"
# query_emb = embedding_manager.generate_embeddings([query_text])
# print(f"Query embedding shape: {query_emb.shape}")

# # Direct ChromaDB query
# direct_results = vectoer_Store.collection.query(
#     query_embeddings=[query_emb[0].tolist()],
#     n_results=3
# )

# print(f"\n=== Direct Query Results ===")
# print(f"Number of results: {len(direct_results['documents'][0]) if direct_results['documents'] else 0}")

# if direct_results['documents'] and direct_results['documents'][0]:
#     for i, (doc, dist) in enumerate(zip(direct_results['documents'][0], direct_results['distances'][0])):
#         print(f"\n--- Result {i+1} ---")
#         print(f"Distance: {dist:.4f}")
#         print(f"Similarity: {1-dist:.4f}")
#         print(f"Content preview: {doc[:150]}")